# D1.8 · Threat intel sub-lane

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.7 · Drift monitoring](https://spbreed.github.io/cyber-commons/lessons/D1.7.html)**.

| | |
|---|---|
| Tools used | MISP, OpenCTI, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Build a synthesis loop that must cite or abstain.

**Why a security engineer needs it.** Unsourced confidence in synthesis loops. The control it builds is: provenance discipline; refuse claims without a source.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Two intel questions, not one: how adversaries are using AI, and who is coming for the AI you run. Most programmes track the first because it is written about, and the second is the one that reaches your estate.

> **At CyberTravels.** Two intel questions for CyberTravels: how adversaries use agents, and who is coming for CyberTravels. The second is the one that reaches the booking API.

## 2 · The framework

```
   two intel questions, only one of which is well covered

   how adversaries use AI        who is coming for the AI you run
   +---------------------+       +-----------------------------+
   | written about a lot |       | your models, agents, MCP    |
   | mostly capability   |       | servers, eval corpora       |
   +---------------------+       +-----------------------------+
                                       the one that reaches you
```

Threat intel is judged by exactly one thing: **how many detections came out of
it.** Everything else — feed volume, report quality, briefing frequency — is
input, not outcome.

An indicator is actionable when two things are true:

- it is a **type you can match on** (a host, a hash, a specific technique with a
  concrete precondition), and
- its **confidence justifies the false-positive cost** of the rule it becomes.

A narrative about adversary trends is not intelligence you can operate. It may
be genuinely useful for planning and it should not be counted as detection
coverage, because counting it that way makes a programme look covered when it is
not.

## 3 · Demo — a feed, converted

In [ ]:
import time
from dataclasses import dataclass

@dataclass
class Indicator:
    value: str; kind: str; source: str; confidence: float

FEED = [
 Indicator("collect.example.com", "host", "vendor-a", 0.95),
 Indicator("169.254.169.254", "host", "internal-research", 0.99),
 Indicator("a1b2c3d4e5f6", "hash", "vendor-b", 0.72),
 Indicator("pastebin.example", "host", "vendor-a", 0.55),
 Indicator("adversaries increasingly use agentic tooling", "narrative", "blog", 0.40),
 Indicator("agents reading ~/.aws/credentials", "technique", "internal-ir", 0.88),
 Indicator("threat actor GOLDEN-OTTER is targeting fintech", "narrative", "vendor-c", 0.60),
]
CONF_FLOOR = 0.70
MATCHABLE = {"host", "hash", "technique"}

def actionable(i):
    if i.kind not in MATCHABLE:
        return False, f"{i.kind} is not matchable in telemetry"
    if i.confidence < CONF_FLOOR:
        return False, f"confidence {i.confidence} below floor {CONF_FLOOR}"
    return True, "convertible to a rule"

print(f"{'indicator':46s}{'kind':11s}{'conf':>6}  verdict")
print("-" * 88)
for i in FEED:
    ok, why = actionable(i)
    print(f"{i.value[:44]:46s}{i.kind:11s}{i.confidence:>6.2f}  "
          f"{'RULE' if ok else 'drop'} — {why}")

In [ ]:
@dataclass
class Rule:
    name: str; severity: str; match: object; response: str

def to_rules(feed):
    rules = []
    for i in feed:
        ok, _ = actionable(i)
        if not ok: continue
        sev = "critical" if i.confidence > 0.9 else "high"
        if i.kind == "host":
            m = (lambda v: (lambda e: v in e.get("target", "")))(i.value)
            resp = f"block egress, revoke the agent's token, hunt back 30d ({i.source})"
        elif i.kind == "hash":
            m = (lambda v: (lambda e: v == e.get("hash", "")))(i.value)
            resp = f"quarantine the artefact, check the supply chain ({i.source})"
        else:
            m = (lambda: (lambda e: "/.aws/" in e.get("target", "")))()
            resp = f"revoke, rotate the cloud role, audit reads ({i.source})"
        rules.append(Rule(f"intel:{i.kind}:{i.value[:26]}", sev, m, resp))
    return rules

rules = to_rules(FEED)
print(f"{len(FEED)} indicators → {len(rules)} deployable rules "
      f"({len(rules)/len(FEED):.0%} conversion)\n")
for r in rules:
    print(f"   [{r.severity:8s}] {r.name}")

## 4 · Where it breaks — conversion is only the first of three numbers

In [ ]:
EVENTS = [
 {"actor": "patch-agent", "target": "https://collect.example.com/x"},
 {"actor": "triage-agent", "target": "https://api.github.com/repos"},
 {"actor": "patch-agent", "target": "/home/app/.aws/credentials"},
 {"actor": "svc-etl", "target": "/data/export.csv"},
 {"actor": "build-agent", "hash": "a1b2c3d4e5f6"},
]
fired = [(r, e) for r in rules for e in EVENTS if r.match(e)]
print("alerts generated from the feed:")
for r, e in fired:
    print(f"   [{r.severity}] {r.name}")
    print(f"        actor={e['actor']}  → {r.response}")

ACTIONED = 2      # of those alerts, how many led to an action
print(f"\nthe three numbers that matter:")
print(f"   indicators received : {len(FEED)}")
print(f"   rules deployed      : {len(rules)}  ({len(rules)/len(FEED):.0%} of the feed)")
print(f"   alerts fired        : {len(fired)}")
print(f"   alerts actioned     : {ACTIONED}  ({ACTIONED/max(len(fired),1):.0%})")
print("\nThe third number is the one that decides whether the subscription renews.")

## 5 · The control — agent-specific intel is mostly internal

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">intel source</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">converts to a detection</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">why</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">your own incidents</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>100%</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the technique that worked against you</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">your red team (C1)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>90%</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">attack-suite results become detections directly</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">your drift monitor (D1.7)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>80%</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">baseline changes are leading indicators</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">vendor advisories</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">50%</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">useful for the supply chain (C2.5)</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">commercial feed</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">30%</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">generic indicators; little agent-specific content yet</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The highest-converting sources are all internal. For agentic threats the intel programme is mostly a feedback loop out of C1 and D1.7, not a purchase.</div>

## What you just proved

Four of seven indicators convert to rules — the two narratives and the low-confidence host are dropped with reasons. The rules fire on three of five events with concrete responses. The three-number summary shows a 57% conversion rate and 67% of alerts actioned, and the source table ranks internal sources highest.

## Your turn

Compute your own three numbers for last quarter: indicators received, rules deployed, alerts actioned. The ratio between the first and third is the honest value of the programme.

---

**Next → [D1.9 · Detections whose subject is the agent platform](https://spbreed.github.io/cyber-commons/lessons/D1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*